# Cascade Mask R-CNN on the RTS challenge — RGB baseline

Fine-tunes a COCO-pretrained **Cascade Mask R-CNN** (detectron2) on retrogressive thaw
slump chips, using the **red/green/blue bands only**, and writes a validated
`submission.json`.

### Before you run this

1. Run `prepare_png_dataset.py` locally to build `data_png/` from the release, then
   **upload that folder as a Kaggle Dataset** and attach it here. It is ~64 MB, against
   ~547 MB for the raw `.npz` release.
2. Settings → **Accelerator: GPU**, and **Internet: On** for the first run (detectron2
   has to be built from source).
3. Leave `SMOKE_TEST = True` for the first run. It proves the whole pipeline in about
   fifteen minutes. Only then set it to `False` for the real ~2–3 hour run.

The design decisions behind every setting here are argued in
`detectron2_training_guide.md`.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG - every knob for this notebook lives here.
# ---------------------------------------------------------------------------
SMOKE_TEST = True          # short run that proves the pipeline end to end
RUN_TRAIN  = True
RUN_INFER  = True
SEED       = 42

# Solver - sized for ~2-3 h on one Kaggle T4 (~40 epochs over 642 training chips)
IMS_PER_BATCH     = 2
BASE_LR           = 0.0025
MAX_ITER          = 15000
STEPS             = (10000, 13500)
WARMUP_ITERS      = 500
CHECKPOINT_PERIOD = 2000
EVAL_PERIOD       = 2000
NUM_WORKERS       = 2

# Inference
DETECTIONS_PER_IMAGE = 20
SCORE_THRESH_TEST    = 0.05

CONFIG_YAML = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
OUTPUT_DIR  = "/kaggle/working/output"

if SMOKE_TEST:
    MAX_ITER, STEPS, WARMUP_ITERS = 200, (150,), 50
    CHECKPOINT_PERIOD = EVAL_PERIOD = 100
    print("SMOKE_TEST is on - short run to prove the pipeline, not a competitive model")

print(f"iters={MAX_ITER}  batch={IMS_PER_BATCH}  lr={BASE_LR}")

In [ ]:
import importlib
import subprocess
import sys

import numpy as np
import torch

print("torch :", torch.__version__, "| cuda:", torch.version.cuda,
      "| gpu:", torch.cuda.is_available())
print("numpy :", np.__version__)

if int(np.__version__.split(".")[0]) >= 2:
    print("WARNING: numpy 2.x - detectron2's extensions need numpy<2.")
    print("         Run: pip install 'numpy<2'  then RESTART the kernel.")

try:
    import detectron2
    import detectron2._C                      # the compiled extension - the real test
    print("detectron2:", detectron2.__version__, "(already present)")
except ImportError:
    print("building detectron2 from source, ~10 min ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/facebookresearch/detectron2.git"])
    importlib.invalidate_caches()
    import detectron2
    import detectron2._C
    print("detectron2:", detectron2.__version__, "(built)")

In [ ]:
import csv
import json
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_test_loader,
                             build_detection_train_loader)
from detectron2.data import transforms as T
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultTrainer, HookBase
from detectron2.evaluation import COCOEvaluator
from detectron2.modeling import build_model
from detectron2.utils.logger import setup_logger
from detectron2.utils.visualizer import Visualizer

setup_logger()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def find_data_root() -> Path:
    """Locate data_png/ on Kaggle or locally, by its annotations file."""
    roots = [Path("/kaggle/input"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        for hit in sorted(root.glob("**/annotations/instances_train_fold.json")):
            return hit.parent.parent
    raise FileNotFoundError(
        "data_png not found. Attach it as a Kaggle Dataset, or run "
        "prepare_png_dataset.py locally."
    )


DATA = find_data_root()
print("data root:", DATA)

In [ ]:
for name, fold in (("rts_train", "train"), ("rts_val", "val")):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)
    register_coco_instances(
        name,
        {},
        str(DATA / "annotations" / f"instances_{fold}_fold.json"),
        str(DATA / "train"),
    )
    MetadataCatalog.get(name).thing_classes = ["rts"]

train_dicts = DatasetCatalog.get("rts_train")
val_dicts = DatasetCatalog.get("rts_val")
print(f"train: {len(train_dicts):>4} images, "
      f"{sum(len(d['annotations']) for d in train_dicts):>5} instances")
print(f"val  : {len(val_dicts):>4} images, "
      f"{sum(len(d['annotations']) for d in val_dicts):>5} instances")

In [ ]:
d = train_dicts[0]
print("record keys     :", sorted(d))
print("annotation keys :", sorted(d["annotations"][0]))
print("category_id     :", d["annotations"][0]["category_id"], " (remapped 1 -> 0)")
print("bbox_mode       :", d["annotations"][0]["bbox_mode"])

img = cv2.imread(d["file_name"])[:, :, ::-1]
assert img is not None, f"could not read {d['file_name']}"
assert img.shape[:2] == (d["height"], d["width"]), "PNG size disagrees with the COCO record"
for a in d["annotations"]:
    assert a["segmentation"]["size"] == [d["height"], d["width"]], "RLE size mismatch"
    assert a["category_id"] == 0, "category_id must be 0 internally"

n_inst = sum(len(x["annotations"]) for x in train_dicts + val_dicts)
assert n_inst == 1783, f"expected 1783 instances across both folds, got {n_inst}"
print(f"\nall checks passed - {n_inst} instances across both folds")

vis = Visualizer(img, metadata=MetadataCatalog.get("rts_train"), scale=3.0)
plt.figure(figsize=(11, 7))
plt.imshow(vis.draw_dataset_dict(d).get_image())
plt.axis("off")
plt.title(f"{Path(d['file_name']).name} - {len(d['annotations'])} instance(s)")
plt.show()

In [ ]:
def build_cfg() -> "CfgNode":
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(CONFIG_YAML))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CONFIG_YAML)

    cfg.DATASETS.TRAIN = ("rts_train",)
    cfg.DATASETS.TEST = ("rts_val",)
    cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1        # propagates to all three cascade heads
    cfg.INPUT.MASK_FORMAT = "bitmask"          # labels are RLE, not polygons

    cfg.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
    cfg.SOLVER.BASE_LR = BASE_LR
    cfg.SOLVER.MAX_ITER = MAX_ITER
    cfg.SOLVER.STEPS = STEPS
    cfg.SOLVER.WARMUP_ITERS = WARMUP_ITERS
    cfg.SOLVER.CHECKPOINT_PERIOD = CHECKPOINT_PERIOD
    cfg.SOLVER.AMP.ENABLED = True
    cfg.TEST.EVAL_PERIOD = EVAL_PERIOD

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
    cfg.TEST.DETECTIONS_PER_IMAGE = DETECTIONS_PER_IMAGE

    # Deliberately left at their defaults:
    #   INPUT.FORMAT = "BGR" + COCO PIXEL_MEAN -> read_image does the RGB->BGR flip
    #   MIN_SIZE_TRAIN = (640..800) -> upsamples these small chips ~4.2x
    #   ANCHOR_GENERATOR.SIZES = 32..512 -> measured to match at that upsampling
    # Shrink MIN_SIZE_TRAIN and you must shrink the anchors by the same factor.

    cfg.OUTPUT_DIR = OUTPUT_DIR
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg


cfg = build_cfg()
print("roi heads     :", cfg.MODEL.ROI_HEADS.NAME, "| classes:", cfg.MODEL.ROI_HEADS.NUM_CLASSES)
print("input format  :", cfg.INPUT.FORMAT, "| pixel mean:", cfg.MODEL.PIXEL_MEAN)
print("min_size_train:", cfg.INPUT.MIN_SIZE_TRAIN)
print("anchor sizes  :", cfg.MODEL.ANCHOR_GENERATOR.SIZES)

In [ ]:
TRAIN_AUGS = [
    T.ResizeShortestEdge(cfg.INPUT.MIN_SIZE_TRAIN, cfg.INPUT.MAX_SIZE_TRAIN, "choice"),
    T.RandomFlip(horizontal=True, vertical=False),
    T.RandomFlip(horizontal=False, vertical=True),
]


class LossEvalHook(HookBase):
    """Periodically report the validation loss - DefaultTrainer never does."""

    def __init__(self, period, model, loader):
        self._period = period
        self._model = model
        self._loader = loader

    def _do_loss_eval(self):
        was_training = self._model.training
        self._model.train()
        totals, n = {}, 0
        with torch.no_grad():
            for batch in self._loader:
                for k, v in self._model(batch).items():
                    totals[k] = totals.get(k, 0.0) + float(v)
                n += 1
        self._model.train(was_training)
        means = {f"val_{k}": v / max(n, 1) for k, v in totals.items()}
        self.trainer.storage.put_scalars(val_total_loss=sum(means.values()), **means)

    def after_step(self):
        nxt = self.trainer.iter + 1
        if self._period > 0 and nxt % self._period == 0 and nxt != self.trainer.max_iter:
            self._do_loss_eval()


class RTSTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(cfg, is_train=True, augmentations=TRAIN_AUGS)
        return build_detection_train_loader(cfg, mapper=mapper)

    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        return build_detection_test_loader(
            cfg, dataset_name, mapper=DatasetMapper(cfg, is_train=False))

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=output_folder or cfg.OUTPUT_DIR)

    def build_hooks(self):
        hooks = super().build_hooks()
        loss_loader = build_detection_test_loader(
            self.cfg, self.cfg.DATASETS.TEST[0],
            mapper=DatasetMapper(self.cfg, is_train=True, augmentations=[
                T.ResizeShortestEdge(self.cfg.INPUT.MIN_SIZE_TEST,
                                     self.cfg.INPUT.MAX_SIZE_TEST)]))
        hooks.insert(-1, LossEvalHook(self.cfg.TEST.EVAL_PERIOD, self.model, loss_loader))
        return hooks

In [ ]:
if RUN_TRAIN:
    trainer = RTSTrainer(cfg)
    trainer.resume_or_load(resume=True)
    trainer.train()
    print("training finished ->", os.path.join(cfg.OUTPUT_DIR, "model_final.pth"))
else:
    print("RUN_TRAIN is False - skipping training")

In [ ]:
def encode_binary_mask(mask):
    """Encode an H x W binary mask as compressed COCO RLE (JSON-safe)."""
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


def load_trained_model(weights=None):
    cfg_i = build_cfg()
    cfg_i.MODEL.WEIGHTS = weights or os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    model = build_model(cfg_i)
    DetectionCheckpointer(model).load(cfg_i.MODEL.WEIGHTS)
    model.eval()
    return cfg_i, model


def predict(model, cfg_i, dataset, top_k=10):
    """Run the model over a dataset name or a list of dicts -> COCO results."""
    loader = build_detection_test_loader(
        dataset if isinstance(dataset, list) else cfg_i,
        **({} if isinstance(dataset, list) else {"dataset_name": dataset}),
        mapper=DatasetMapper(cfg_i, is_train=False),
    )
    results = []
    with torch.no_grad():
        for batch in loader:
            for inp, out in zip(batch, model(batch)):
                inst = out["instances"].to("cpu")
                order = inst.scores.argsort(descending=True)[:top_k]
                for i in order.tolist():
                    results.append({
                        "image_id": int(inp["image_id"]),
                        "category_id": 1,
                        "segmentation": encode_binary_mask(inst.pred_masks[i].numpy()),
                        "score": float(inst.scores[i]),
                    })
    return results

In [ ]:
OFFICIAL_MAXDETS = [1, 5, 10]
OFFICIAL_AREA_RNG = [[0, 1e10], [0, 300], [300, 2000], [2000, 1e10]]
OFFICIAL_AREA_LBL = ["all", "small", "medium", "large"]


def score_official(gt_json_path, predictions):
    """COCO segm AP using the challenge's maxDets and area ranges."""
    if not predictions:
        print("no predictions - nothing to score")
        return None
    coco_gt = COCO(str(gt_json_path))
    coco_dt = coco_gt.loadRes(list(predictions))
    ev = COCOeval(coco_gt, coco_dt, "segm")
    ev.params.maxDets = OFFICIAL_MAXDETS
    ev.params.areaRng = OFFICIAL_AREA_RNG
    ev.params.areaRngLbl = OFFICIAL_AREA_LBL
    ev.evaluate()
    ev.accumulate()

    area_i = ev.params.areaRngLbl.index("all")
    det_i = ev.params.maxDets.index(10)
    prec = ev.eval["precision"][:, :, :, area_i, det_i]
    primary = float(np.mean(prec[prec > -1])) if (prec > -1).any() else -1.0
    print(f"AP @[IoU=0.50:0.95 | area=all | maxDets=10] = {primary:.4f}   <- ranking metric")
    return primary


if RUN_INFER:
    cfg_i, model = load_trained_model()
    val_preds = predict(model, cfg_i, "rts_val", top_k=10)
    print(f"{len(val_preds)} predictions over {len(val_dicts)} validation images")
    score_official(DATA / "annotations" / "instances_val_fold.json", val_preds)

In [ ]:
if RUN_INFER:
    with open(DATA / "test_manifest.csv", newline="") as f:
        rows = list(csv.DictReader(f))

    test_dicts = [{
        "file_name": str(DATA / "test" / f"{r['public_id']}.png"),
        "image_id": int(r["image_id"]),
        "height": int(r["height"]),
        "width": int(r["width"]),
    } for r in rows]
    print(f"{len(test_dicts)} test cips")

    test_preds = predict(model, cfg_i, test_dicts, top_k=10)
    out_path = "/kaggle/working/submission.json"
    with open(out_path, "w") as f:
        json.dump(test_preds, f)
    print(f"wrote {out_path}  ({len(test_preds)} predictions)")

    sizes = {int(r["image_id"]): (int(r["height"]), int(r["width"])) for r in rows}
    seen = {}
    for i, p in enumerate(test_preds):
        assert set(p) == {"image_id", "category_id", "segmentation", "score"}, i
        assert p["image_id"] in sizes, f"unknown image_id {p['image_id']}"
        assert p["category_id"] == 1, "category_id must be 1"
        assert 0.0 <= p["score"] <= 1.0 and np.isfinite(p["score"]), "bad score"
        assert p["segmentation"]["size"] == list(sizes[p["image_id"]]), "RLE size mismatch"
        assert mask_utils.decode(p["segmentation"]).shape == sizes[p["image_id"]]
        seen[p["image_id"]] = seen.get(p["image_id"], 0) + 1

    crowded = {k: v for k, v in seen.items() if v > 10}
    print(f"validation=ok  images_with_predictions={len(seen)}/{len(rows)}")
    if crowded:
        print(f"note: {len(crowded)} images exceed maxDets=10")

## Next steps

Once `SMOKE_TEST = False` has produced a real baseline, in rough order of expected value:

1. **Input resolution.** `INPUT.MIN_SIZE_TRAIN` is the biggest lever on both memory and
   accuracy. If you lower it, scale `MODEL.ANCHOR_GENERATOR.SIZES` by the same factor —
   they are one coupled decision.
2. **Learning rate and schedule.** `0.0025` is the linear-scaling upper bound; fine-tuning
   642 images often prefers less.
3. **Augmentation.** Rotations are as defensible as the flips already here, for the same
   reason — nadir imagery has no canonical orientation.
4. **`MODEL.BACKBONE.FREEZE_AT`.** Default `2`. On 642 images, freezing early layers is a
   reasonable regulariser; worth an ablation against `0`.
5. **More channels.** Only once the RGB pipeline is solid. That means leaving the PNG path
   and reinstating a custom mapper — see the appendix of `detectron2_training_guide.md`.

Compare runs on the number from cell 11, not on the `COCOEvaluator` output printed during
training: those use COCO's area bins and `maxDets`, not this challenge's.